In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

#### 3.1) Загрузка исходных данных

In [2]:
train = pd.read_parquet("train.parquet")
validation = pd.read_parquet("validation.parquet")
test = pd.read_parquet("test.parquet")

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Train: (413754, 21)
Validation: (83496, 21)
Test: (87274, 21)


In [3]:
print(
    "Train:",
    train["created_at"].min(),
    "->",
    train["created_at"].max()
)

print(
    "Validation:",
    validation["created_at"].min(),
    "->",
    validation["created_at"].max()
)

print(
    "Test:",
    test["created_at"].min(),
    "->",
    test["created_at"].max()
)

Train: 2016-07-01 00:00:00 -> 2017-12-07 00:00:00
Validation: 2017-12-08 00:00:00 -> 2018-03-27 00:00:00
Test: 2018-03-28 00:00:00 -> 2018-08-28 00:00:00


#### 3.2) Бейзлайн

В качестве бейзлайна возьмем следующую идею: будем предсказывать то же значение, которое было в тот же день год назад. Это позволит использовать сезонный паттерн в данных.

In [4]:
def make_daily_orders(df):
    daily = (
        df.groupby("created_at")["increment_id"]
          .nunique()
          .sort_index()
    )

    full_dates = pd.date_range(
        daily.index.min(),
        daily.index.max(),
        freq="D"
    )

    return daily.reindex(full_dates, fill_value=0)

In [5]:
train_daily = make_daily_orders(train)
val_daily = make_daily_orders(validation)

print(train_daily.head())
print()
print(val_daily.head())

2016-07-01    447
2016-07-02    198
2016-07-03    120
2016-07-04    182
2016-07-05    114
Freq: D, Name: increment_id, dtype: int64

2017-12-08    616
2017-12-09    359
2017-12-10    258
2017-12-11    378
2017-12-12    367
Freq: D, Name: increment_id, dtype: int64


In [6]:
predictions = []

for date in val_daily.index:
    previous_year_date = date - pd.DateOffset(years=1)

    if previous_year_date in train_daily.index:
        prediction = train_daily.loc[previous_year_date]
    else:
        prediction = np.nan

    predictions.append(prediction)

val_pred = pd.Series(
    predictions,
    index=val_daily.index,
    name="prediction"
)

In [7]:
comparison = pd.DataFrame({
    "actual": val_daily,
    "prediction": val_pred
})

comparison.head(10)

,actual,prediction
2017-12-08,616,362
2017-12-09,359,370
2017-12-10,258,332
2017-12-11,378,229
2017-12-12,367,201
2017-12-13,442,382
2017-12-14,315,386
2017-12-15,582,434
2017-12-16,261,409
2017-12-17,175,362


In [8]:
y_true = comparison["actual"]
y_pred = comparison["prediction"]

mae = mean_absolute_error(y_true, y_pred)

rmse = np.sqrt(
    mean_squared_error(y_true, y_pred)
)

r2 = r2_score(y_true, y_pred)

print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²:   {r2:.3f}")

MAE:  344.48
RMSE: 746.63
R²:   -0.070


Сезонный бейзлайн показал, насколько хорошо целевой показатель можно прогнозировать, опираясь только на повторяющиеся временные закономерности. Его результат можно использовать как отправную точку для сравнения с более сложными моделями. Если бустинг и другие модели заметно превосходят сезонный бейзлайн на validation- и test-выборках, это означает, что дополнительные признаки действительно содержат полезную информацию и улучшают качество прогноза. Если же разница небольшая, то основная часть предсказуемости объясняется сезонностью, а усложнение модели даёт ограниченный прирост качества.

Попробуем ещё один бейзлайн: будем предсказывать среднее число заказов в день.

In [9]:
mean_daily_orders = train_daily.mean()

print("Среднее количество заказов в день:", round(mean_daily_orders, 2))

Среднее количество заказов в день: 545.73


In [10]:
mean_pred = pd.Series(
    mean_daily_orders,
    index=val_daily.index,
    name="prediction"
)

mean_comparison = pd.DataFrame({
    "actual": val_daily,
    "prediction": mean_pred
})

mean_comparison.head()

,actual,prediction
2017-12-08,616,545.727619
2017-12-09,359,545.727619
2017-12-10,258,545.727619
2017-12-11,378,545.727619
2017-12-12,367,545.727619


In [11]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

y_true = mean_comparison["actual"]
y_pred = mean_comparison["prediction"]

mae_mean = mean_absolute_error(y_true, y_pred)

rmse_mean = np.sqrt(
    mean_squared_error(y_true, y_pred)
)

r2_mean = r2_score(y_true, y_pred)

print(f"MAE:  {mae_mean:.2f}")
print(f"RMSE: {rmse_mean:.2f}")
print(f"R²:   {r2_mean:.3f}")

MAE:  413.59
RMSE: 721.74
R²:   -0.000


При сравнении двух бейзлайнов предсказание среднего показывает лучший результат по RMSE и R²: RMSE составляет 721.74 против 746.63 у сезонного бейзлайна, а R² близок к нулю, тогда как у сезонной модели он отрицательный (-0.070). Это означает, что сезонный бейзлайн в целом не превосходит простое предсказание среднего.

При этом сезонный бейзлайн имеет более низкий MAE — 344.48 против 413.59, то есть его типичная абсолютная ошибка меньше. Однако более высокий RMSE указывает на наличие более крупных отдельных ошибок. В целом, если основной метрикой является RMSE, модель предсказания среднего является более сильным бейзлайном.